# Classroom Batch Analysis
This notebook performs a bulk analysis on a "virtual classroom" (a cohort of ~50 students) to identify aggregate learning gaps and curriculum risks.

In [7]:
# 1. Imports and Setup
import json
import pandas as pd
import time
import random
from typing import List, Dict
from google import genai
from google.genai import types

# Import project utilities
from utils.constants import GEMINI_API_KEY
from utils.dataset import load_joined_datasets, get_best_attempts
import lib.llm_batch_analyzer
from lib.llm_batch_analyzer import format_submissions, clean_json_response, create_system_instruction

# Configure Target Scope
# We examine a sequence of String Manipulation problems to see progress
# Problems: 31 (Repeat End), 32 (Plus Out), 33 (Mix String), 34 (Zip Zap)
TARGET_PROBLEMS = [31, 32, 33, 34] 
lib.llm_batch_analyzer.FOCUS_PROBLEMS = TARGET_PROBLEMS 

# Initialize Gemini Client
client = genai.Client(api_key=GEMINI_API_KEY)
print(f"Environment Configured. Target Problems: {TARGET_PROBLEMS}")

Environment Configured. Target Problems: [31, 32, 33, 34]


In [8]:
# 2. Data Loading & Cohort Selection
# We simulate a "Classroom" by selecting 50 students who submitted code for MULTIPLE assignments.

# Config
CLASS_SIZE = 50

print("Loading dataset...")
df = load_joined_datasets()

if df is not None:
    # Use best attempts
    best_df = get_best_attempts(df)
    
    # 1. Filter for the target assignments
    focused_df = best_df[best_df['ProblemID'].isin(TARGET_PROBLEMS)]
    
    # 2. Identify students who completed ALL (or most) targets
    # We want valid history, so we look for students with at least 4 entries in this subset
    submission_counts = focused_df.groupby('SubjectID').size()
    qualified_students = submission_counts[submission_counts >= len(TARGET_PROBLEMS)].index.tolist()
    
    print(f"Students with history across problems {TARGET_PROBLEMS}: {len(qualified_students)}")
    
    # 3. Select Random Cohort
    if len(qualified_students) > 0:
        selected_ids = random.sample(qualified_students, min(CLASS_SIZE, len(qualified_students)))
        
        # 4. Create Batches (Dict of SubjectID -> List of Submissions)
        # We assume one best attempt per problem per student
        student_batches = []
        for s_id in selected_ids:
            # Get their work, sort by ProblemID to show progression
            work = focused_df[focused_df['SubjectID'] == s_id].sort_values('ProblemID').to_dict('records')
            student_batches.append(work)
            
        print(f"Created Virtual Classroom of {len(student_batches)} students.")
        print(f"Each student has {len(student_batches[0])} submissions analyzed.")
    else:
        print("No qualified students found for this problem set.")
        student_batches = []
else:
    print("Error: Dataset not found.")

Loading dataset...
Loading datasets...
Main table: 201,570 rows
CodeState table: 69,627 rows
Subject table: 381 rows

Joining datasets...
Joined dataset: 191,584 rows
Datasets joined successfully.
Columns in the joined dataset:
Index(['Order', 'SubjectID', 'ToolInstances', 'ServerTimestamp',
       'ServerTimezone', 'CourseID', 'CourseSectionID', 'TermID',
       'AssignmentID', 'ProblemID', 'Attempt', 'CodeStateID',
       'IsEventOrderingConsistent', 'EventType', 'Score', 'Compile.Result',
       'CompileMessageType', 'CompileMessageData', 'EventID', 'ParentEventID',
       'SourceLocation', 'Code', 'X-Grade'],
      dtype='object')
Best attempts: 15,375 rows
  Unique students: 372
  Unique problems: 50
Students with history across problems [31, 32, 33, 34]: 297
Created Virtual Classroom of 50 students.
Each student has 4 submissions analyzed.


In [9]:
# 3. Define Analysis Agent

def get_evaluator_instruction() -> str:
    # This now pulls specific problem descriptions for [32, 33, 34, 35]
    base_instruction = create_system_instruction()
    
    strict_instruction = """
    
    STRICT BATCH OUTPUT RULES:
    1. 'knowledge_gaps' must be a list of objects:
       { "category": "TAG", "description": "text..." }
       
    2. 'future_predictions' must be a list of objects:
       { "topic_tag": "TAG", "risk_explanation": "text..." }
       
    PERMITTED TAGS:
    [Loop, NestedLoop, String, Array, Logic, Condition, Method, Math, Indexing, Comparison]
    """
    return base_instruction + strict_instruction

def analyze_student_history(submissions: List[Dict]) -> Dict:
    """
    Hits the LLM with a BATCH of submissions for a single student.
    The LLM sees the progression and identifies persistent gaps.
    """
    # format_submissions handles list logic
    formatted_input = format_submissions(submissions)
    system_instr = get_evaluator_instruction()
    
    try:
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=formatted_input,
            config=types.GenerateContentConfig(
                system_instruction=system_instr,
                temperature=0.3,
                response_mime_type="application/json"
            )
        )
        if response.text:
            return json.loads(clean_json_response(response.text))
    except Exception as e:
        print(f"Error: {e}")
        return None

In [ ]:
# 4. Batch Execution
# We run the analysis for the whole class history.

class_results = []
print(f"Starting Aggregate Analysis for {len(student_batches)} students...")

total_start = time.time()

for i, sub_batch in enumerate(student_batches):
    s_id = sub_batch[0]['SubjectID']
    if i % 5 == 0: print(f"Processing student {i+1}/{len(student_batches)}...")
    
    # Analyze the history
    result = analyze_student_history(sub_batch)
    
    if result and 'student_analysis' in result:
        # result['student_analysis'] is a list of analysis objects (one per problem)
        analyses = result['student_analysis']
        
        # Aggregate Risks across the 4 assignments
        # Use simple set to collect unique risks identified across their history
        all_risks = set()
        all_gaps = set()
        
        for entry in analyses:
            for p in entry.get('future_predictions', []):
                all_risks.add(p.get('topic_tag', 'Unknown'))
            for g in entry.get('knowledge_gaps', []):
                all_gaps.add(g.get('category', 'Unknown'))
        
        class_results.append({
            "Student_ID": s_id,
            "Predicted_Risks": list(all_risks),
            "Identified_Gaps": list(all_gaps),
            "Risk_Count": len(all_risks)
        })

duration = time.time() - total_start
print(f"\nBatch Complete in {duration:.2f} seconds.")

Starting Aggregate Analysis for 50 students...
Processing student 1/50...


Processing student 6/50...


In [ ]:
# 5. Aggregate Results & Visualization
# Generate the Heatmap Data

results_df = pd.DataFrame(class_results)

# A. Risk Frequency (Course Level View)
all_risks = [risk for risks in results_df['Predicted_Risks'] for risk in risks]
risk_counts = pd.Series(all_risks).value_counts()

print(f"--- TOP PREDICTED RISKS (Based on History of {len(TARGET_PROBLEMS)} Assignments) ---")
print(risk_counts)

# B. Student Risk Matrix (Heatmap Prep)
# We explode the list of risks into rows to pivot
exploded_df = results_df.explode('Predicted_Risks')
risk_matrix = pd.crosstab(exploded_df['Student_ID'], exploded_df['Predicted_Risks'])

print("\n--- CLASSROOM RISK MATRIX (Sample) ---")
display(risk_matrix.head(10))

# Optional: Simple Visualization
try:
    import matplotlib.pyplot as plt
    
    plt.figure(figsize=(10, 6))
    risk_counts.plot(kind='bar', color='salmon')
    plt.title(f"Forecasted Class Struggles (Consolidated Analysis of probs {TARGET_PROBLEMS})")
    plt.ylabel("Number of At-Risk Students")
    plt.xlabel("Topic")
    plt.xticks(rotation=45)
    plt.show()
except ImportError:
    print("Matplotlib not installed.")

KeyError: 'Predicted_Risks'